# AI Assignment 2026 — Classical Reasoning Agent
Pipeline: **Natural language / FOL text → symbolic representation → classical reasoning → True/False**

* **RuleTaker-style KBs**: rule parser + unification + forward chaining (with negation-as-failure) + proof traces
* **FOLIO (first-order logic)**: own FOL parser (with syntax repair) → CNF/Skolemisation → **own resolution prover**; Z3 is used only as a time-boxed fallback
* **CounterBench**: causal-rule parser → structural causal model (do-intervention), world enumeration for observations
* No generative model is used anywhere. Run all cells top to bottom (CPU is enough; GPU is not needed).

In [1]:
!pip -q install datasets z3-solver reportlab

In [2]:
import re, time, json, os, itertools, heapq, random, shutil, zipfile
from collections import defaultdict, deque, Counter
import pandas as pd
import z3

RANDOM_SEED = 0
random.seed(RANDOM_SEED)
OUT_DIR = "submission"
os.makedirs(OUT_DIR, exist_ok=True)
METRICS = {}          # everything the report prints comes from here (no hard-coded numbers)
print("ready")

ready


## 1. RuleTaker engine (facts + rules → forward chaining with provenance)

In [3]:
PRON = {"someone","something","they","it","everyone","everybody","somebody","anyone","anything"}
ENT = (r"(?:[Tt]he\s+[A-Za-z]+(?:\s+[A-Za-z]+)?|[Ss]omeone|[Ss]omething|[Tt]hey|[Ii]t|"
       r"[Ee]veryone|[Ee]verybody|[Ss]omebody|[A-Z][a-z]+)")

def ent(t):
    t = t.strip().lower()
    if t in PRON: return "?x"
    t = re.sub(r"^(the|a|an)\s+", "", t)
    return re.sub(r"\s+", "_", t)

def vnorm(v):                      # generic verb normaliser: sees->see, chases->chase
    v = v.lower()
    return v[:-1] if v.endswith("s") and not v.endswith("ss") and len(v) > 3 else v

def parse_atom(text, default_subj=None):
    """-> (predicate, args, negated) or None. Variables are '?x'."""
    t = text.strip().rstrip(".").strip()
    m = re.fullmatch(rf"({ENT})\s+(?:is|are)\s+(not\s+)?([A-Za-z_-]+)", t)
    if m:
        s, neg, p = m.groups()
        return (p.lower(), (ent(s),), bool(neg))
    m = re.fullmatch(rf"({ENT})\s+(?:(?:does|do)\s+(not)\s+)?([a-z]+)\s+({ENT})", t)
    if m and m.group(3) not in ("is", "are", "be"):
        s, neg, v, o = m.groups()
        return (vnorm(v), (ent(s), ent(o)), bool(neg))
    if default_subj is not None:                       # "... and not cold" / "... and big"
        m = re.fullmatch(r"(?:(?:is|are)\s+)?(not\s+)?([A-Za-z_-]+)", t)
        if m:
            neg, p = m.groups()
            return (p.lower(), (default_subj,), bool(neg))
    return None

def parse_rule(s):
    s = s.strip().rstrip(".").strip()
    m = re.fullmatch(r"If\s+(.+?)\s+then\s+(.+)", s, flags=re.I)
    if m:
        ctext, htext = m.groups()
        def atoms(txt):
            out, subj = [], None
            for part in re.split(r"\s+and\s+", txt):
                a = parse_atom(part, subj)
                if a is None: return None
                out.append(a); subj = a[1][0]
            return out
        conds, heads = atoms(ctext), atoms(htext)
        if conds is None or heads is None: return None
        cvars = {a for c in conds for a in c[1] if a.startswith("?")}
        if any(a.startswith("?") and a not in cvars for h in heads for a in h[1]): return None
        return (tuple(conds), tuple(heads))
    m = re.fullmatch(r"(?:[Aa]ll\s+)?([A-Za-z]+(?:(?:\s*,\s*|\s+and\s+)[A-Za-z]+)*)\s+(?:people|things)\s+"
                     r"(?:are|is)\s+(not\s+)?([A-Za-z_-]+)", s)
    if m:
        adjs, neg, concl = m.groups()
        conds = tuple((a.lower(), ("?x",), False) for a in re.split(r"\s*,\s*|\s+and\s+", adjs) if a)
        return (conds, ((concl.lower(), ("?x",), bool(neg)),))
    return None

def parse_context(context):
    pos, neg, rules, bad = set(), set(), [], []
    for sent in re.split(r"(?<=[.!?])\s+", context.strip()):
        sent = sent.strip()
        if not sent: continue
        if re.match(r"^If\b", sent, flags=re.I) or re.search(r"\s(people|things)\s+(are|is)\s", sent) and not parse_atom(sent):
            r = parse_rule(sent)
            (rules.append(r) if r else bad.append(sent)); continue
        a = parse_atom(sent)
        if a is None:
            r = parse_rule(sent)
            (rules.append(r) if r else bad.append(sent)); continue
        (neg if a[2] else pos).add((a[0], a[1]))
    return pos, neg, rules, bad

def _bind(pattern, actual, b):
    nb = dict(b)
    for p, a in zip(pattern, actual):
        if p.startswith("?"):
            if p in nb and nb[p] != a: return None
            nb[p] = a
        elif p != a: return None
    return nb

def _join(conds, idx, b, used):
    if not conds:
        yield b, used; return
    pred, args, _ = conds[0]
    for fargs in idx.get(pred, ()):
        if len(fargs) != len(args): continue
        nb = _bind(args, fargs, b)
        if nb is not None:
            yield from _join(conds[1:], idx, nb, used + [(pred, fargs)])

def _naf(negs, b, facts, domain):
    results = [b]
    for pred, args, _ in negs:
        nxt = []
        for bb in results:
            fv = list(dict.fromkeys(a for a in args if a.startswith("?") and a not in bb))
            for combo in itertools.product(domain, repeat=len(fv)):
                b2 = {**bb, **dict(zip(fv, combo))}
                if (pred, tuple(b2.get(a, a) for a in args)) not in facts:   # negation as failure
                    nxt.append(b2)
        results = nxt
    return results

def forward_chain(pos, neg, rules):
    """Forward chaining to a fixpoint. Returns (facts, provenance)."""
    facts = set(pos)
    prov = {f: ("given",) for f in facts}
    domain = sorted({a for f in (pos | neg) for a in f[1]})
    dset = set(domain)
    changed = True
    while changed:
        changed = False
        for ri, (conds, heads) in enumerate(rules):
            pcs = [c for c in conds if not c[2]]
            ncs = [c for c in conds if c[2]]
            idx = defaultdict(list)
            for p, a in facts: idx[p].append(a)
            matches = []
            for b, used in _join(pcs, idx, {}, []):
                for b2 in _naf(ncs, b, facts, domain):
                    matches.append((b2, used))
            for b2, used in matches:
                for h in heads:
                    if h[2]: continue                       # negative heads are not used (closed world)
                    fact = (h[0], tuple(b2.get(a, a) for a in h[1]))
                    if any(a.startswith("?") for a in fact[1]): continue
                    if fact not in facts:
                        facts.add(fact); prov[fact] = ("rule", ri, list(used)); changed = True
                        for a in fact[1]:
                            if a not in dset: dset.add(a); domain.append(a)
    return facts, prov

def fact_str(f):
    p, a = f
    return f"{p}({', '.join(a)})"

def explain(fact, prov, depth=0, seen=None):
    seen = seen if seen is not None else set()
    pad = "  " * depth
    if fact in seen: return pad + fact_str(fact) + "  (already shown)\n"
    seen.add(fact)
    pr = prov.get(fact)
    if pr is None: return pad + fact_str(fact) + "  [not derived]\n"
    if pr[0] == "given": return pad + fact_str(fact) + "  [given fact]\n"
    out = pad + fact_str(fact) + f"  [derived by rule #{pr[1]+1}]\n"
    for u in pr[2]: out += explain(u, prov, depth + 1, seen)
    return out

def predict_rt(context, question, trace=False):
    """Returns 'entailment' / 'not entailment' (and a proof trace if requested)."""
    pos, neg, rules, bad = parse_context(context)
    facts, prov = forward_chain(pos, neg, rules)
    q = parse_atom(question)
    if q is None:
        return ("not entailment", "question could not be parsed") if trace else "not entailment"
    fact = (q[0], q[1])
    holds = fact in facts
    ans = (not holds) if q[2] else holds          # closed-world: "not P" holds iff P cannot be derived
    label = "entailment" if ans else "not entailment"
    if not trace: return label
    if q[2]:
        t = (f"Query: NOT {fact_str(fact)}\n" +
             (f"{fact_str(fact)} is derivable, so the negated query fails:\n" + explain(fact, prov)
              if holds else f"{fact_str(fact)} cannot be derived from the facts and rules (negation as failure) -> True"))
    else:
        t = f"Query: {fact_str(fact)}\n" + (explain(fact, prov) if holds else "not derivable -> False")
    return label, t

# --- sanity test from the assignment text
_ctx = "Bob is young. Bob is kind. If someone is young, then they are smart. If someone is smart and kind then they are successful."
print(predict_rt("Bob is young. Bob is kind. If someone is young then they are smart. If someone is smart and kind then they are successful.",
                 "Bob is successful.", trace=True)[1])

Query: successful(bob)
successful(bob)  [derived by rule #2]
  smart(bob)  [derived by rule #1]
    young(bob)  [given fact]
  kind(bob)  [given fact]



## 2. First-order logic engine (FOLIO)
FOL text → tokenizer → recursive-descent parser (with parenthesis repair) → NNF → Skolemisation → CNF →
**binary resolution with factoring (set-of-support, unit preference)**. If the resolution budget is exhausted
(or equality is involved) a time-boxed Z3 check is used as a fallback.

In [4]:
TOK = re.compile(r"\s*(∀|∃|¬|∧|∨|⊕|→|↔|⟷|⇔|⇒|≠|=|\(|\)|,|[^\s()∀∃¬∧∨⊕→↔⟷⇔⇒≠=,]+)")
def tokenize_fol(s):
    s = s.replace("​", "").strip()
    pos, out = 0, []
    while pos < len(s):
        m = TOK.match(s, pos)
        if not m: break
        out.append(m.group(1)); pos = m.end()
    return out

class FOLParseError(Exception): pass

class FOLParser:
    def __init__(self, text):
        self.t = tokenize_fol(text); self.i = 0
    def peek(self): return self.t[self.i] if self.i < len(self.t) else None
    def eat(self, x=None):
        tok = self.peek()
        if tok is None or (x is not None and tok != x):
            raise FOLParseError(f"expected {x!r} got {tok!r}")
        self.i += 1; return tok
    def parse(self):
        f = self.p_iff(set())
        if self.peek() is not None: raise FOLParseError(f"trailing token {self.peek()!r}")
        return f
    def p_iff(self, b):
        l = self.p_imp(b)
        while self.peek() in ("↔", "⟷", "⇔"):
            self.eat(); l = ("iff", l, self.p_imp(b))
        return l
    def p_imp(self, b):
        l = self.p_xor(b)
        if self.peek() in ("→", "⇒"):
            self.eat(); return ("imp", l, self.p_imp(b))          # right-assoc
        return l
    def p_xor(self, b):
        l = self.p_or(b)
        while self.peek() == "⊕":
            self.eat(); l = ("xor", l, self.p_or(b))
        return l
    def p_or(self, b):
        l = self.p_and(b)
        while self.peek() == "∨":
            self.eat(); l = ("or", l, self.p_and(b))
        return l
    def p_and(self, b):
        l = self.p_un(b)
        while self.peek() == "∧":
            self.eat(); l = ("and", l, self.p_un(b))
        return l
    def p_un(self, b):
        t = self.peek()
        if t == "¬":
            self.eat(); return ("not", self.p_un(b))
        if t in ("∀", "∃"):
            self.eat()
            names = [self.eat()]
            while self.peek() == ",": self.eat(); names.append(self.eat())
            nb = set(b) | set(names)
            while self.peek() in ("∀", "∃") and False: pass
            body = self.p_un(nb)
            for v in reversed(names): body = ("all" if t == "∀" else "ex", v, body)
            return body
        if t == "(":
            save = self.i
            self.eat()
            f = self.p_iff(b)
            self.eat(")")
            if self.peek() in ("=", "≠"):               # parenthesised term in equality (rare)
                raise FOLParseError("paren term")
            return f
        return self.p_atom(b)
    def p_term(self, b):
        name = self.eat()
        if name in "()," or name in ("∀","∃","¬","∧","∨","⊕","→","↔","=","≠"): raise FOLParseError(f"bad term {name!r}")
        if self.peek() == "(":
            self.eat(); args = []
            if self.peek() != ")":
                args.append(self.p_term(b))
                while self.peek() == ",": self.eat(); args.append(self.p_term(b))
            self.eat(")")
            return ("fn", name, tuple(args))
        return ("var", name) if name in b else ("fn", name, ())
    def p_atom(self, b):
        t = self.p_term(b)
        if self.peek() in ("=", "≠"):
            op = self.eat(); r = self.p_term(b)
            e = ("eq", t, r)
            return ("not", e) if op == "≠" else e
        return ("atom", t[1], t[2])

def repair_parens(s):
    d = s.count("(") - s.count(")")
    if d > 0: return s + ")" * d
    if d < 0:
        for _ in range(-d):
            k = s.rfind(")"); s = s[:k] + s[k+1:]
    return s

def parse_fol(s):
    try:
        return FOLParser(s).parse()
    except (FOLParseError, IndexError):
        return FOLParser(repair_parens(s)).parse()

def has_eq(f):
    if f[0] == "eq": return True
    if f[0] == "atom": return False
    return any(has_eq(x) for x in f[1:] if isinstance(x, tuple) and x and isinstance(x[0], str) and x[0] in
               ("eq","atom","not","and","or","imp","iff","xor","all","ex"))

In [5]:
_ctr = itertools.count(1)
def fresh_var(): return ("V", next(_ctr))

def nnf(f, pos=True):
    k = f[0]
    if k in ("atom", "eq"): return f if pos else ("not", f)
    if k == "not": return nnf(f[1], not pos)
    if k == "and": return (("and" if pos else "or"), nnf(f[1], pos), nnf(f[2], pos))
    if k == "or":  return (("or" if pos else "and"), nnf(f[1], pos), nnf(f[2], pos))
    if k == "imp": return nnf(("or", ("not", f[1]), f[2]), pos)
    if k == "iff": return nnf(("and", ("imp", f[1], f[2]), ("imp", f[2], f[1])), pos)
    if k == "xor": return nnf(("and", ("or", f[1], f[2]), ("not", ("and", f[1], f[2]))), pos)
    if k == "all": return (("all" if pos else "ex"), f[1], nnf(f[2], pos))
    if k == "ex":  return (("ex" if pos else "all"), f[1], nnf(f[2], pos))
    raise ValueError(k)

def subst_term(t, env):
    if t[0] == "var": return env.get(t[1], t)
    if t[0] == "fn": return ("fn", t[1], tuple(subst_term(a, env) for a in t[2]))
    return t   # ("V", n)

_sk = itertools.count(1)
def skolemize(f, env, univ):
    k = f[0]
    if k == "atom": return ("atom", f[1], tuple(subst_term(a, env) for a in f[2]))
    if k == "eq":   return ("eq", subst_term(f[1], env), subst_term(f[2], env))
    if k == "not":  return ("not", skolemize(f[1], env, univ))
    if k in ("and", "or"): return (k, skolemize(f[1], env, univ), skolemize(f[2], env, univ))
    if k == "all":
        v = fresh_var()
        return skolemize(f[2], {**env, f[1]: v}, univ + [v])
    if k == "ex":
        sk = ("fn", f"sk{next(_sk)}", tuple(univ))
        return skolemize(f[2], {**env, f[1]: sk}, univ)

class TooBig(Exception): pass

def to_clauses(f, cap=3000):
    k = f[0]
    if k == "and":
        a, b = to_clauses(f[1], cap), to_clauses(f[2], cap)
        if len(a) + len(b) > cap: raise TooBig
        return a + b
    if k == "or":
        a, b = to_clauses(f[1], cap), to_clauses(f[2], cap)
        if len(a) * len(b) > cap: raise TooBig
        return [x + y for x in a for y in b]
    if k == "not":
        g = f[1]
        return [[(False, g[0] == "eq" and "=" or g[1], g[2] if g[0] == "atom" else (g[1], g[2]))]]
    if k == "atom": return [[(True, f[1], f[2])]]
    if k == "eq":   return [[(True, "=", (f[1], f[2]))]]
    raise ValueError(k)

def rename_clause(cl):
    m = {}
    def r(t):
        if t[0] == "V":
            if t not in m: m[t] = fresh_var()
            return m[t]
        return ("fn", t[1], tuple(r(a) for a in t[2]))
    return tuple((s, p, tuple(r(a) for a in args)) for s, p, args in cl)

def clausify(formulas):
    out = []
    for f in formulas:
        g = skolemize(nnf(f), {}, [])
        for cl in to_clauses(g):
            # remove duplicate literals, drop tautologies
            lits = list(dict.fromkeys(cl))
            if any((not s, p, a) in lits for s, p, a in lits): continue
            out.append(rename_clause(lits))
    return out

def walk(t, s):
    while t[0] == "V" and t in s: t = s[t]
    return t
def occurs(v, t, s):
    t = walk(t, s)
    if t == v: return True
    return t[0] == "fn" and any(occurs(v, a, s) for a in t[2])
def unify(a, b, s):
    a, b = walk(a, s), walk(b, s)
    if a == b: return s
    if a[0] == "V":
        if occurs(a, b, s): return None
        return {**s, a: b}
    if b[0] == "V":
        if occurs(b, a, s): return None
        return {**s, b: a}
    if a[1] != b[1] or len(a[2]) != len(b[2]): return None
    for x, y in zip(a[2], b[2]):
        s = unify(x, y, s)
        if s is None: return None
    return s
def apply(t, s):
    t = walk(t, s)
    return t if t[0] == "V" else ("fn", t[1], tuple(apply(a, s) for a in t[2]))
def apply_lit(l, s): return (l[0], l[1], tuple(apply(a, s) for a in l[2]))
def term_size(t): return 1 if t[0] == "V" else 1 + sum(term_size(a) for a in t[2])
def clause_weight(cl): return sum(1 + sum(term_size(a) for a in l[2]) for l in cl)

def norm_clause(lits):
    lits = list(dict.fromkeys(lits))
    sl = set(lits)
    if any((not s, p, a) in sl for s, p, a in lits): return None       # tautology
    return rename_clause(lits)

def resolvents(c1, c2):
    for i, l1 in enumerate(c1):
        for j, l2 in enumerate(c2):
            if l1[0] != l2[0] and l1[1] == l2[1] and len(l1[2]) == len(l2[2]):
                s = {}
                for x, y in zip(l1[2], l2[2]):
                    s = unify(x, y, s)
                    if s is None: break
                if s is None: continue
                lits = [apply_lit(l, s) for k, l in enumerate(c1) if k != i] + \
                       [apply_lit(l, s) for k, l in enumerate(c2) if k != j]
                yield lits
def factors(c):
    for i in range(len(c)):
        for j in range(i + 1, len(c)):
            a, b = c[i], c[j]
            if a[0] == b[0] and a[1] == b[1] and len(a[2]) == len(b[2]):
                s = {}
                for x, y in zip(a[2], b[2]):
                    s = unify(x, y, s)
                    if s is None: break
                if s is not None:
                    yield [apply_lit(l, s) for k, l in enumerate(c) if k != j]

def resolution_prove(premises, goal, time_limit=1.5, max_clauses=6000, max_weight=45):
    """premises: list of AST, goal: AST. Returns ('proved'|'saturated'|'unknown', stats)."""
    t0 = time.time()
    usable = clausify(premises)
    sos = clausify([("not", goal)])
    stats = {"generated": 0}
    pruned = False
    seen = set()
    heap, cnt = [], itertools.count()
    def push(c):
        nonlocal pruned
        if c is None: return
        if clause_weight(c) > max_weight: pruned = True; return
        if len(c) == 0: raise StopIteration
        heapq.heappush(heap, (len(c) + clause_weight(c) * 0.1, next(cnt), c))
    try:
        for c in sos:
            if len(c) == 0: return "proved", stats
            push(c)
        used = list(usable)
        while heap:
            if time.time() - t0 > time_limit or stats["generated"] > max_clauses: return "unknown", stats
            _, _, g = heapq.heappop(heap)
            m = {}
            def cv(t):
                if t[0] == "V": return ("V", m.setdefault(t, len(m)))
                return ("fn", t[1], tuple(cv(a) for a in t[2]))
            variants = tuple((sg, p, tuple(cv(a) for a in args)) for sg, p, args in g)
            if variants in seen: continue
            seen.add(variants)
            new = []
            for other in used + [g]:
                for r in resolvents(g, other): new.append(r)
            for r in factors(g): new.append(r)
            used.append(g)
            for r in new:
                stats["generated"] += 1
                if len(r) == 0: return "proved", stats
                push(norm_clause(r))
        return ("unknown" if pruned else "saturated"), stats
    except StopIteration:
        return "proved", stats

In [6]:
def to_z3(f, U, env, decl):
    k = f[0]
    def T(t):
        if t[0] == "var": return env[t[1]]
        key = (t[1], len(t[2]))
        if key not in decl:
            decl[key] = z3.Const(t[1] + "_c", U) if not t[2] else z3.Function(t[1] + "_f", *([U] * len(t[2])), U)
        return decl[key] if not t[2] else decl[key](*[T(a) for a in t[2]])
    if k == "atom":
        key = ("P:" + f[1], len(f[2]))
        if key not in decl:
            decl[key] = z3.Bool(f[1] + "_p") if not f[2] else z3.Function(f[1] + "_p", *([U] * len(f[2])), z3.BoolSort())
        return decl[key] if not f[2] else decl[key](*[T(a) for a in f[2]])
    if k == "eq":  return T(f[1]) == T(f[2])
    if k == "not": return z3.Not(to_z3(f[1], U, env, decl))
    if k == "and": return z3.And(to_z3(f[1], U, env, decl), to_z3(f[2], U, env, decl))
    if k == "or":  return z3.Or(to_z3(f[1], U, env, decl), to_z3(f[2], U, env, decl))
    if k == "imp": return z3.Implies(to_z3(f[1], U, env, decl), to_z3(f[2], U, env, decl))
    if k == "iff": return to_z3(f[1], U, env, decl) == to_z3(f[2], U, env, decl)
    if k == "xor": return z3.Xor(to_z3(f[1], U, env, decl), to_z3(f[2], U, env, decl))
    if k in ("all", "ex"):
        v = z3.Const(f"v{next(_ctr)}", U)
        body = to_z3(f[2], U, {**env, f[1]: v}, decl)
        return z3.ForAll([v], body) if k == "all" else z3.Exists([v], body)

def z3_entails(premises, goal, timeout_ms=3000):
    U = z3.DeclareSort("U"); decl = {}
    s = z3.Solver(); s.set("timeout", timeout_ms)
    for p in premises: s.add(to_z3(p, U, {}, decl))
    s.add(z3.Not(to_z3(goal, U, {}, decl)))
    r = s.check()
    return "proved" if r == z3.unsat else ("not_proved" if r == z3.sat else "unknown")

def predict_folio(premises_text, conclusion_text, trace=False):
    """Returns True/False (entailed or not). Strings: newline-separated premises."""
    info = {}
    try:
        lines = [l for l in re.split(r"\n+", premises_text.strip()) if l.strip()]
        prem = [parse_fol(l) for l in lines]
        goal = parse_fol(conclusion_text)
    except Exception as e:
        info["engine"] = "parse-failure"; info["error"] = str(e)
        return (False, info) if trace else False
    eq = any(has_eq(x) for x in prem + [goal])
    res = "unknown"
    if not eq:
        try:
            res, st = resolution_prove(prem, goal)
            info["engine"] = "resolution"; info["stats"] = st
        except (TooBig, RecursionError):
            res = "unknown"
    if res == "unknown":
        try:
            res = z3_entails(prem, goal)
            info["engine"] = "z3-fallback"
        except Exception as e:
            res = "unknown"; info["engine"] = "z3-error"
    out = (res == "proved")
    info["result"] = res
    return (out, info) if trace else out

# --- quick tests
_P = "∀x (Drinks(x) → Dependent(x))\nDrinks(rina)"
print(predict_folio(_P, "Dependent(rina)", True))
print(predict_folio(_P, "¬Dependent(rina)", True))
print(predict_folio("∀x (Cat(x) → Animal(x))\n∃x Cat(x)", "∃x Animal(x)", True))
print(predict_folio("∀x (A(x) ⊕ B(x))\nA(a)", "¬B(a)", True))
print(predict_folio("a = b\nP(a)", "P(b)", True))
print(predict_folio("∀x (P(x) → Q(x)", "Q(a)", True))

(True, {'engine': 'resolution', 'stats': {'generated': 2}, 'result': 'proved'})
(False, {'engine': 'resolution', 'stats': {'generated': 0}, 'result': 'saturated'})
(True, {'engine': 'resolution', 'stats': {'generated': 2}, 'result': 'proved'})
(True, {'engine': 'resolution', 'stats': {'generated': 3}, 'result': 'proved'})
(True, {'engine': 'z3-fallback', 'result': 'proved'})
(False, {'engine': 'resolution', 'stats': {'generated': 1}, 'result': 'saturated'})


## 3. CounterBench engine (causal rules → structural causal model)
Causal sentences are parsed into rules (`causes`, `AND`/`OR` connective, signed `effects`).
A structural causal model is evaluated in topological order; an **intervention** `do(X=v)` overrides a
variable's structural equation. When observations are given, we enumerate all exogenous-root worlds
(abduction), keep those consistent with the observations and then apply the intervention in each of them.

In [7]:
def sgn(lit):
    lit = lit.strip()
    m = re.match(r"^not\s+(.+)$", lit, flags=re.I)
    return (m.group(1).strip(), False) if m else (lit, True)

def _split_effects(t):
    return [e.strip() for e in re.split(r"\s*,\s*(?:and\s+)?|\s+and\s+", t.strip()) if e.strip()]

def parse_cb_rules(given):
    text = str(given).strip()
    text = re.sub(r"^\s*We know that\s+", "", text, flags=re.I)
    text = re.sub(r"\.?\s*We observed\b.*$", "", text, flags=re.I).strip(" .")
    text = re.sub(r",?\s*[A-Za-z][A-Za-z-]*\s*~\s*Bern\s*\(\s*[\d.]+\s*\)", "", text, flags=re.I).strip(" ,.")
    text = re.sub(r"\b(or\s+(?:not\s+)?[A-Za-z][A-Za-z-]*)\s+together\s+(causes?)\b", r"\1 \2", text, flags=re.I)
    text = re.sub(r",\s*and\s+([A-Za-z]+),\s*and\s+\1\s+causes?\s+", r" and \1. \1 causes ", text, flags=re.I)
    lit = r"(?:not\s+)?[A-Za-z][A-Za-z-]*"
    split_pat = rf"\.\s*|,\s*(?=(?:and\s+)?{lit}(?:\s+(?:or|and)\s+{lit})?(?:\s+together)?\s+causes?\s+)"
    rules = []
    for cl in re.split(split_pat, text, flags=re.I):
        cl = re.sub(r"^and\s+", "", cl.strip().strip(" ."), flags=re.I)
        if not cl: continue
        m = re.fullmatch(r"(.+?)\s+and\s+(.+?)(?:\s+together)?\s+causes?\s+(.+)", cl, flags=re.I)
        if m:
            a, b, e = m.groups(); rules.append(((a.strip(), b.strip()), "and", _split_effects(e))); continue
        m = re.fullmatch(r"(.+?)\s+or\s+(.+?)\s+causes?\s+(.+)", cl, flags=re.I)
        if m:
            a, b, e = m.groups(); rules.append(((a.strip(), b.strip()), "or", _split_effects(e))); continue
        m = re.fullmatch(r"(.+?)\s+causes?\s+(.+)", cl, flags=re.I)
        if m:
            a, e = m.groups(); rules.append(((a.strip(),), "and", _split_effects(e)))
    return rules

def parse_cb_obs(given):
    m = re.search(r"\bWe observed\s+(.+?)\s*\.?\s*$", str(given), flags=re.I)
    if not m: return []
    out = []
    for it in re.split(r"\s+and\s+", m.group(1).strip().rstrip("."), flags=re.I):
        it = it.strip()
        mm = re.fullmatch(r"([A-Za-z][A-Za-z-]*)\s+not\s+occurs?", it, flags=re.I)
        if mm: out.append(("not " + mm.group(1))); continue
        mm = re.fullmatch(r"([A-Za-z][A-Za-z-]*)\s+occurs?", it, flags=re.I)
        out.append(mm.group(1) if mm else it)
    return out

def parse_cb_question(q):
    q = str(q).strip()
    m = re.fullmatch(r"Assume\s+(.+?),\s*and\s+based\s+on\s+this\s+assumption,\s*further\s+suppose\s+(.+?)\.\s*Would\s+(.+?)\s+occur\??", q, flags=re.I)
    if m:
        a, b, t = m.groups(); return {"target": t.strip(), "do": [a.strip(), b.strip()], "kind": "nested"}
    m = re.fullmatch(r"Would\s+(.+?)\s+occur\s+if\s+(.+?)\s+instead\s+of\s+(.+?)\??", q, flags=re.I)
    if m:
        t, i, _ = m.groups(); return {"target": t.strip(), "do": re.split(r"\s+and\s+", i.strip()), "kind": "single"}
    m = re.fullmatch(r"Would\s+(.+?)\s+occur\s+if\s+(.+?)\??", q, flags=re.I)
    if m:
        t, i = m.groups(); return {"target": t.strip(), "do": re.split(r"\s+and\s+", i.strip()), "kind": "joint"}
    return None

class SCM:
    def __init__(self, rules):
        self.rules = rules
        self.inc = defaultdict(list)          # var -> [(causes, connective, effect_value)]
        parents = defaultdict(set); vs = set()
        for causes, conn, effects in rules:
            cv = [sgn(c)[0] for c in causes]; vs.update(cv)
            for e in effects:
                v, val = sgn(e); vs.add(v)
                self.inc[v].append((causes, conn, val))
                parents[v].update(p for p in cv if p != v)
        self.vars, self.parents = vs, parents
        self.roots = sorted(v for v in vs if not self.inc[v])
        # topological order
        indeg = {v: len(parents[v]) for v in vs}; ch = defaultdict(set)
        for v in vs:
            for p in parents[v]: ch[p].add(v)
        q = deque(sorted(v for v in vs if indeg[v] == 0)); order = []
        while q:
            v = q.popleft(); order.append(v)
            for c in sorted(ch[v]):
                indeg[c] -= 1
                if indeg[c] == 0: q.append(c)
        if len(order) != len(vs): raise ValueError("cycle")
        self.order = order
    def run(self, roots=None, do=None):
        roots, do = roots or {}, do or {}
        val = {}
        for v in self.order:
            if v in do: val[v] = do[v]; continue
            if not self.inc[v]: val[v] = roots.get(v, False); continue
            pol = self.inc[v][0][2]
            trig = False
            for causes, conn, _ in self.inc[v]:
                cs = [val[sgn(c)[0]] == sgn(c)[1] for c in causes]
                if (all(cs) if conn == "and" else any(cs)): trig = True
            val[v] = pol if trig else (not pol)
        for v, x in do.items(): val[v] = x
        return val

def predict_cb(given, question, nested_mode="default"):
    """Returns (bool, info). True == the target event occurs under the counterfactual."""
    rules = parse_cb_rules(given); obs = parse_cb_obs(given); q = parse_cb_question(question)
    if q is None or not rules: return False, {"error": "parse"}
    m = SCM(rules)
    do = {}
    for l in q["do"]:
        v, x = sgn(l); do[v] = x
    tgt = q["target"]
    if obs:                                  # abduction over exogenous roots
        o = [sgn(x) for x in obs]
        roots = m.roots
        if len(roots) > 14: roots = roots[:14]
        compat = []
        for mask in range(2 ** len(roots)):
            r = {v: bool(mask >> i & 1) for i, v in enumerate(roots)}
            f = m.run(r)
            if all(f.get(v) == x for v, x in o): compat.append(r)
        if not compat:
            return False, {"worlds": 0}
        ys = [m.run(r, do).get(tgt, False) for r in compat]
        return (sum(ys) * 2 > len(ys)), {"worlds": len(compat), "yes": sum(ys)}
    if q["kind"] == "nested" and nested_mode != "default":
        ys = [m.run({v: bool(mk >> i & 1) for i, v in enumerate(m.roots[:12])}, do).get(tgt, False)
              for mk in range(2 ** min(len(m.roots), 12))]
        return (all(ys) if nested_mode == "all" else any(ys)), {"worlds": len(ys)}
    f = m.run({}, do)                        # roots default to False when nothing is observed
    return bool(f.get(tgt, False)), {"values": f}

# --- tests taken from the examples printed in the dev data
print(predict_cb("We know that Ziklo causes Blaf, Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo.", "Would Lumbo occur if not Ziklo instead of Ziklo?"))
print(predict_cb("We know that Blaf causes Ziklo, Ziklo or Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo. Blaf~ Bern(0.2). We observed Trune", "Would Lumbo occur if not Ziklo instead of Ziklo?"))

(False, {'values': {'Ziklo': False, 'Blaf': False, 'Trune': False, 'Vork': False, 'Lumbo': False}})
(True, {'worlds': 1, 'yes': 1})


## 4. Unified reasoning agent + submission generator
`predict_any` inspects the input, routes it to the right symbolic engine and always returns **True / False**.

In [8]:
FOL_SYMS = re.compile(r"[∀∃→∧∨¬⊕↔]")
CTX_KEYS = ["premises-FOL", "premises_fol", "context", "knowledge_base", "kb", "theory", "given_info", "premises"]
Q_KEYS = ["conclusion-FOL", "conclusion_fol", "question", "query", "hypothesis", "conclusion"]

def predict_any(context, question, return_info=False):
    context, question = str(context), str(question)
    if FOL_SYMS.search(context) or FOL_SYMS.search(question):
        out, info = predict_folio(context, question, trace=True); dom = "folio"
    elif re.search(r"\bcauses?\b", context) and re.match(r"\s*(Would|Assume)\b", question):
        out, info = predict_cb(context, question, nested_mode=METRICS.get("nested_mode", "default")); dom = "counterbench"
    else:
        out = predict_rt(context, question) == "entailment"; info = {}; dom = "ruletaker"
    return (out, dom, info) if return_info else out

def predict_row(row):
    ctx = next((row[k] for k in CTX_KEYS if k in row and isinstance(row[k], str) and row[k].strip()), "")
    q = next((row[k] for k in Q_KEYS if k in row and isinstance(row[k], str) and row[k].strip()), "")
    try:
        return bool(predict_any(ctx, q))
    except Exception:
        return False                       # never crash on a malformed example

def make_submission(df, id_col=None, path=f"{OUT_DIR}/submission.csv", label_col="predicted_label"):
    """df: DataFrame of unlabeled test queries. Writes id + predicted label (True/False) and prints timing."""
    id_col = id_col or next((c for c in df.columns if c.lower() in ("id", "example_id", "question_id", "idx")), None)
    ids = df[id_col].tolist() if id_col else list(range(len(df)))
    t0 = time.time()
    preds = [predict_row(r) for r in df.to_dict("records")]
    el = time.time() - t0
    out = pd.DataFrame({"id": ids, label_col: ["True" if p else "False" for p in preds]})
    out.to_csv(path, index=False)
    print(f"Wrote {path}: {len(out)} rows, inference time {el:.3f}s ({el/max(len(out),1)*1000:.2f} ms/example)")
    return out

## 5. Experiments on the public datasets
RuleTaker (held-out `test` split sample), FOLIO (validation), CounterBench (alpha + backdoor).
Accuracy and wall-clock inference time are measured here and reused in the report.

In [9]:
from datasets import load_dataset
from huggingface_hub import hf_hub_download
ruletaker = load_dataset("hitachi-nlp/ruletaker")
folio = load_dataset("tasksource/folio")
def _cb(fn):
    return json.load(open(hf_hub_download(repo_id="CounterBench/CounterBench", filename=fn, repo_type="dataset"), encoding="utf-8"))
cb_alpha, cb_backdoor = _cb("data_balanced_alpha_V1.json"), _cb("data_balanced_backdoor_V2.json")
print({k: len(v) for k, v in ruletaker.items()}, len(folio["validation"]), len(cb_alpha), len(cb_backdoor))

README.md:   0%|          | 0.00/762 [00:00<?, ?B/s]

data/train-00000-of-00001-27dba3dca49e9a(…): reconstructing file:   0%|          |  0.00B / 70.6MB            

data/train-00000-of-00001-27dba3dca49e9a(…): downloading bytes:           |  0.00B            

data/test-00000-of-00001-39464af5362e1a3(…): reconstructing file:   0%|          |  0.00B / 21.3MB            

data/test-00000-of-00001-39464af5362e1a3(…): downloading bytes:           |  0.00B            

data/dev-00000-of-00001-8f5fadfabddb3440(…): reconstructing file:   0%|          |  0.00B / 9.74MB            

... (8 outputs left out)

Generating validation split: 0 examples [00:00, ? examples/s]

data_balanced_alpha_V1.json:   0%|          | 0.00/557k [00:00<?, ?B/s]

data_balanced_backdoor_V2.json:   0%|          | 0.00/117k [00:00<?, ?B/s]

{'train': 480152, 'test': 151911, 'dev': 75872} 203 1000 200


In [10]:
N_RT = min(5000, len(ruletaker["test"]), len(ruletaker["dev"]))
rt_test = ruletaker["test"].shuffle(seed=RANDOM_SEED).select(range(N_RT))
rt_dev = ruletaker["dev"].shuffle(seed=RANDOM_SEED).select(range(N_RT))
def eval_rt(ds):
    t0 = time.time(); ok = 0; errs = []; per = defaultdict(lambda: [0, 0])
    for ex in ds:
        p = predict_rt(ex["context"], ex["question"])
        c = (p == ex["label"]); ok += c; per[ex["config"]][0] += c; per[ex["config"]][1] += 1
        if not c and len(errs) < 5: errs.append(ex)
    el = time.time() - t0
    return ok / len(ds), el, {k: v[0] / v[1] for k, v in per.items()}, errs
for name, ds in [("dev", rt_dev), ("test", rt_test)]:
    acc, el, per, errs = eval_rt(ds)
    METRICS[f"rt_{name}"] = dict(acc=acc, n=len(ds), time=el, per_config=per)
    print(f"RuleTaker {name}: acc={acc:.4f} n={len(ds)} time={el:.2f}s ({el/len(ds)*1000:.2f} ms/ex)")
    print("  per config:", {k: round(v, 4) for k, v in sorted(per.items())})
    for e in errs[:2]: print("  ERR:", e["question"], "|", e["label"], "|", e["context"][:200])

RuleTaker dev: acc=0.9998 n=5000 time=4.89s (0.98 ms/ex)
  per config: {'NatLang': 1.0, 'depth-0': 1.0, 'depth-1': 1.0, 'depth-2': 1.0, 'depth-3': 1.0, 'depth-3ext': 1.0, 'depth-3ext-NatLang': 1.0, 'depth-5': 0.9984}
  ERR: The cat is kind. | not entailment | The bald eagle does not like the tiger. The bald eagle sees the rabbit. The bald eagle visits the rabbit. The cat sees the tiger. The rabbit is kind. The rabbit is rough. The rabbit sees the bald eagl
RuleTaker test: acc=0.9996 n=5000 time=2.58s (0.52 ms/ex)
  per config: {'NatLang': 1.0, 'depth-0': 1.0, 'depth-1': 1.0, 'depth-2': 1.0, 'depth-3': 0.9983, 'depth-3ext': 0.9986, 'depth-3ext-NatLang': 1.0, 'depth-5': 1.0}
  ERR: The cat does not like the cat. | entailment | The cat is nice. The cat likes the tiger. The tiger is not red. If someone likes the cat then they are cold. If the cat is round then the cat sees the tiger. If someone likes the tiger then they do no
  ERR: The rabbit sees the cow. | not entailment | The bear is c

In [11]:
def eval_folio(ds, limit=None):
    rows = list(ds)[:limit] if limit else list(ds)
    t0 = time.time(); ok = 0; engines = Counter(); per_label = defaultdict(lambda: [0, 0])
    for ex in rows:
        gold = (str(ex["label"]) == "True")
        pred, info = predict_folio(ex["premises-FOL"], ex["conclusion-FOL"], trace=True)
        engines[info.get("engine")] += 1
        ok += (pred == gold); per_label[str(ex["label"])][0] += (pred == gold); per_label[str(ex["label"])][1] += 1
    el = time.time() - t0
    return ok / len(rows), el, engines, {k: v[0] / v[1] for k, v in per_label.items()}, len(rows)
acc, el, eng, pl, n = eval_folio(folio["validation"])
METRICS["folio"] = dict(acc=acc, n=n, time=el, engines=dict(eng), per_label=pl)
print(f"FOLIO validation (binary: entailed vs not): acc={acc:.4f} n={n} time={el:.1f}s engines={dict(eng)} per-gold-label={pl}")

FOLIO validation (binary: entailed vs not): acc=0.8374 n=203 time=9.6s engines={'resolution': 151, 'parse-failure': 2, 'z3-fallback': 50} per-gold-label={'Uncertain': 0.9855072463768116, 'True': 0.5972222222222222, 'False': 0.9516129032258065}


In [12]:
def yn(x): return str(x).strip().lower() in ("yes", "true", "1")
def eval_cb(rows, mode="default"):
    t0 = time.time(); per = defaultdict(lambda: [0, 0])
    for r in rows:
        p, _ = predict_cb(r["given_info"], r["question"], nested_mode=mode)
        c = (p == yn(r["answer"])); per[r["type"]][0] += c; per[r["type"]][1] += 1
    tot = sum(v[0] for v in per.values()) / sum(v[1] for v in per.values())
    return tot, time.time() - t0, {k: v[0] / v[1] for k, v in per.items()}
nested = [r for r in cb_alpha if r["type"] == "nested"]
cal = {}
for mode in ["default", "all", "any"]:
    cal[mode] = eval_cb(nested, mode)[0]
best = max(cal, key=cal.get); METRICS["nested_mode"] = best; METRICS["nested_calibration"] = cal
print("nested semantics calibration (accuracy on public nested examples):", cal, "-> using", best)
for name, rows in [("alpha", cb_alpha), ("backdoor", cb_backdoor)]:
    acc, el, per = eval_cb(rows, best)
    METRICS[f"cb_{name}"] = dict(acc=acc, n=len(rows), time=el, per_type=per)
    print(f"CounterBench {name}: acc={acc:.4f} n={len(rows)} time={el:.2f}s per type={ {k: round(v,3) for k,v in per.items()} }")

nested semantics calibration (accuracy on public nested examples): {'default': 0.96, 'all': 0.96, 'any': 0.96} -> using default
CounterBench alpha: acc=0.9500 n=1000 time=0.17s per type={'basic': 1.0, 'conditional': 0.88, 'joint': 0.96, 'nested': 0.96}
CounterBench backdoor: acc=0.9000 n=200 time=0.06s per type={'backdoor': 0.9}


In [13]:
mixed = []
for ex in list(rt_test)[:500]: mixed.append({"id": f"rt{len(mixed)}", "context": ex["context"], "question": ex["question"], "gold": ex["label"] == "entailment"})
for ex in list(folio["validation"])[:100]: mixed.append({"id": f"fo{len(mixed)}", "context": ex["premises-FOL"], "question": ex["conclusion-FOL"], "gold": str(ex["label"]) == "True"})
for ex in cb_alpha[::5] + cb_backdoor[::5]: mixed.append({"id": f"cb{len(mixed)}", "context": ex["given_info"], "question": ex["question"], "gold": yn(ex["answer"])})
mdf = pd.DataFrame(mixed)
sub = make_submission(mdf.drop(columns="gold"), id_col="id", path=f"{OUT_DIR}/mixed_public_predictions.csv")
t0 = time.time(); preds = [predict_row(r) for r in mdf.drop(columns="gold").to_dict("records")]; el = time.time() - t0
acc = sum(p == g for p, g in zip(preds, mdf["gold"])) / len(mdf)
METRICS["mixed"] = dict(acc=acc, n=len(mdf), time=el)
print(f"UNIFIED AGENT on mixed public sample: accuracy={acc:.4f} (n={len(mdf)}), inference time={el:.2f}s")

Wrote submission/mixed_public_predictions.csv: 840 rows, inference time 8.388s (9.99 ms/example)
UNIFIED AGENT on mixed public sample: accuracy=0.9643 (n=840), inference time=6.39s


In [14]:
TRACES = []
ex = next(e for e in rt_test if e["label"] == "entailment" and e["config"].startswith("depth-3"))
lab, tr = predict_rt(ex["context"], ex["question"], trace=True)
TRACES.append(("RuleTaker (forward chaining, depth-3)", "Context: " + ex["context"] + "\nQuestion: " + ex["question"] + "\nGold: " + ex["label"] + "\n\n" + tr))
fx = next(e for e in folio["validation"] if str(e["label"]) == "True")
fo, finfo = predict_folio(fx["premises-FOL"], fx["conclusion-FOL"], trace=True)
TRACES.append(("FOLIO (CNF + resolution)", "Premises (FOL):\n" + fx["premises-FOL"] + "\nConclusion: " + fx["conclusion-FOL"] +
               f"\nGold: {fx['label']}\nEngine: {finfo.get('engine')}, resolution result: {finfo.get('result')}, clauses generated: {finfo.get('stats')}"))
cx = cb_backdoor[0]; cp, cinfo = predict_cb(cx["given_info"], cx["question"], METRICS["nested_mode"])
rules = parse_cb_rules(cx["given_info"])
TRACES.append(("CounterBench (SCM + abduction)", f"Given: {cx['given_info']}\nQuestion: {cx['question']}\nParsed rules: {rules}\nObservations: {parse_cb_obs(cx['given_info'])}\n"
               f"Compatible root worlds: {cinfo.get('worlds')}; worlds where target occurs after the intervention: {cinfo.get('yes')}\nPrediction: {'yes' if cp else 'no'} (gold: {cx['answer']})"))
for t, b in TRACES: print("###", t, "\n", b, "\n")

### RuleTaker (forward chaining, depth-3) 
 Context: Bob is white. Bob is young. Charlie is white. Erin is rough. Erin is white. Fiona is green. Fiona is young. If something is rough then it is young. If something is nice then it is green. Smart things are white. If Charlie is not kind then Charlie is smart. Smart, green things are rough. If something is not kind then it is nice.
Question: Bob is white.
Gold: entailment

Query: white(bob)
white(bob)  [given fact]
 

### FOLIO (CNF + resolution) 
 Premises (FOL):
∀x (InThisClub(x) ∧ PerformOftenIn(x, schoolTalentShow) → Attend(x, schoolEvent) ∧ VeryEngagedWith(x, schoolEvent))
∀x (InThisClub(x) → PerformOftenIn(x, schoolTalentShow) ⊕ (InActive(x) ∧ Disinterested(x) ∧ MemberOf(x, community)))
∀x (InThisClub(x) ∧ Chaperone(x, highSchoolDance) → ¬(Studen(x) ∧ AttendSchool(x)))
∀x (InThisClub(x) ∧ (InActive(x) ∧ Disinterested(x) ∧ MemberOf(x, community)) → Chaperone(x, highSchoolDances))
∀x (InThisClub(x) ∧ (YoungChildren(x) ⊕ Teenager(x)) 

In [15]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Preformatted, Table, TableStyle
from reportlab.lib import colors
from xml.sax.saxutils import escape

def build_report(path=f"{OUT_DIR}/report.pdf"):
    ss = getSampleStyleSheet(); body = ss["BodyText"]; mono = ParagraphStyle("m", fontName="Courier", fontSize=7, leading=8.5)
    P = lambda t, s=body: Paragraph(t, s); st = []
    M = METRICS
    st += [P("Classical Reasoning Agent — Report", ss["Title"]), P("Kunal Mishra, M.Tech AI, IIT Jodhpur", body), Spacer(1, 8)]
    st += [P("1. Representation", ss["Heading2"]),
           P("Every input is converted to a symbolic form before any reasoning happens. <b>RuleTaker</b>: facts are ground literals (predicate, args); rules are "
             "(conditions → head) with variable '?x', negated conditions (negation as failure), and relation atoms such as likes(X,Y). "
             "<b>FOLIO</b>: formulas are abstract syntax trees (∀ ∃ ¬ ∧ ∨ ⊕ → ↔ =) which are converted to NNF, Skolemised and put into clause normal form. "
             "<b>CounterBench</b>: causal sentences become rules (causes, AND/OR connective, signed effects) forming a structural causal model; "
             "observations are literals and questions are do-interventions plus a target variable.")]
    st += [P("2. Reasoning algorithms", ss["Heading2"]),
           P("<b>Forward chaining</b> with unification/substitution joins to a fixpoint and a provenance record for every derived fact (used for traces); "
             "the query is answered by membership in the closure (closed-world: a negated query holds iff the atom is not derivable). "
             "<b>Binary resolution with factoring</b>, most-general unifiers with occurs check, set-of-support strategy, "
             "smallest-clause-first (unit preference), tautology removal, variant detection and weight limits. A budget-limited Z3 check is used only as a fallback when resolution is "
             "inconclusive or equality is present; the final answer is True only if entailment is established. <b>Causal inference</b>: topological evaluation of the SCM, "
             "intervention by overriding equations, and abduction by enumerating exogenous-root worlds consistent with observations "
             "(majority over compatible worlds). No generative model is used. Inputs that cannot be parsed default to False and never crash the run.")]
    st += [P("3. Experiments on public datasets", ss["Heading2"])]
    rows = [["Dataset / split", "Examples", "Accuracy", "Time (s)"]]
    for k, nm in [("rt_dev", "RuleTaker dev (sample)"), ("rt_test", "RuleTaker test (sample)"), ("folio", "FOLIO validation (entailed vs not)"),
                  ("cb_alpha", "CounterBench alpha"), ("cb_backdoor", "CounterBench backdoor"), ("mixed", "Unified agent, mixed sample")]:
        d = M[k]; rows.append([nm, str(d["n"]), f"{100*d['acc']:.2f}%", f"{d['time']:.2f}"])
    tb = Table(rows); tb.setStyle(TableStyle([("GRID", (0, 0), (-1, -1), .4, colors.grey), ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey), ("FONTSIZE", (0, 0), (-1, -1), 8)]))
    st += [tb, Spacer(1, 6)]
    st += [P(f"CounterBench per-type accuracy (alpha): {escape(str({k: round(v,3) for k,v in M['cb_alpha']['per_type'].items()}))}. "
             f"Nested-question semantics were selected on the public data: {escape(str(M['nested_calibration']))} → '{M['nested_mode']}'. "
             f"FOLIO engines used: {escape(str(M['folio']['engines']))}. RuleTaker per-config accuracy (test sample): "
             f"{escape(str({k: round(v,3) for k,v in M['rt_test']['per_config'].items()}))}. All numbers are measured by the notebook; none are hard-coded. "
             "FOLIO labels True/False/Uncertain are mapped to the binary task (True = entailed, otherwise False), so 'Uncertain' gold items count as False.")]
    st += [P("4. Reasoning trace examples", ss["Heading2"])]
    for t, b in TRACES:
        st += [P(f"<b>{escape(t)}</b>"), Preformatted(b[:2200], mono), Spacer(1, 5)]
    st += [P("5. Limitations", ss["Heading2"]),
           P("The natural-language parsers target the known dataset templates; unusual phrasing is parsed best-effort. FOLIO needs formula inputs; resolution is "
             "complete but budget-limited, so hard problems fall back to Z3 and, if still undecided, to False. CounterBench nested semantics were calibrated on public data.")]
    SimpleDocTemplate(path, pagesize=A4, leftMargin=36, rightMargin=36, topMargin=36, bottomMargin=36).build(st)
build_report()

readme = f"""Classical Reasoning Agent - AI Assignment 2026
=================================================
Files: reasoning_agent.ipynb (full implementation), report.pdf, submission.csv, README.txt

How to run
 1. Open reasoning_agent.ipynb in Google Colab (CPU is sufficient) and Run all.
 2. The notebook installs z3-solver/datasets/reportlab, loads RuleTaker, FOLIO and CounterBench from Hugging Face,
    prints accuracy and inference time per dataset and for the unified agent, and regenerates report.pdf.
 3. To predict the private test set set TEST_PATH (csv/json/jsonl) in the last cell; it calls make_submission(df) and writes
    submission.csv with columns id and predicted_label (True/False). Context/question columns are auto-detected
    ({', '.join(CTX_KEYS)} / {', '.join(Q_KEYS)}).

Method: symbolic parsing -> forward chaining (RuleTaker), CNF + binary resolution with Z3 as time-boxed fallback (FOLIO),
structural causal model with interventions and abduction (CounterBench). No generative AI is used for any decision.
Measured public results: RuleTaker test {100*METRICS['rt_test']['acc']:.2f}%, FOLIO val {100*METRICS['folio']['acc']:.2f}%,
CounterBench alpha {100*METRICS['cb_alpha']['acc']:.2f}% / backdoor {100*METRICS['cb_backdoor']['acc']:.2f}%.
Libraries: python, pandas, z3-solver, datasets, reportlab.
"""
open(f"{OUT_DIR}/README.txt", "w").write(readme)

1314

In [16]:
TEST_PATH = None          # e.g. "/content/private_test.csv"
if TEST_PATH:
    ext = TEST_PATH.rsplit(".", 1)[-1].lower()
    tdf = pd.read_csv(TEST_PATH) if ext == "csv" else (pd.read_json(TEST_PATH, lines=(ext == "jsonl")))
    make_submission(tdf)                       # writes submission/submission.csv
else:
    shutil.copy(f"{OUT_DIR}/mixed_public_predictions.csv", f"{OUT_DIR}/submission.csv")   # placeholder until the private file is provided
    print("TEST_PATH not set: wrote a public-sample submission.csv as a format example.")

# copy the notebook into the folder and zip (works in Colab; the notebook file name must match)
for cand in ["reasoning_agent.ipynb", "/content/reasoning_agent.ipynb"]:
    if os.path.exists(cand): shutil.copy(cand, f"{OUT_DIR}/reasoning_agent.ipynb"); break
shutil.make_archive("submission", "zip", ".", OUT_DIR)
print("Created submission.zip with:", sorted(os.listdir(OUT_DIR)))

TEST_PATH not set: wrote a public-sample submission.csv as a format example.
Created submission.zip with: ['README.txt', 'mixed_public_predictions.csv', 'report.pdf', 'submission.csv']


In [17]:
# Overall metrics for the Google Form
rt  = METRICS["rt_dev"]            # RuleTaker validation (dev) sample
fo  = METRICS["folio"]             # FOLIO validation
cba, cbb = METRICS["cb_alpha"], METRICS["cb_backdoor"]

cb_n    = cba["n"] + cbb["n"]
cb_acc  = (cba["acc"]*cba["n"] + cbb["acc"]*cbb["n"]) / cb_n
cb_time = cba["time"] + cbb["time"]

accs  = [rt["acc"], fo["acc"], cb_acc]
per_s = [rt["time"]/rt["n"], fo["time"]/fo["n"], cb_time/cb_n]

for name, a, t in zip(["RuleTaker", "FOLIO", "CounterBench"], accs, per_s):
    print(f"{name:13s} accuracy = {100*a:.2f}%   time/sample = {t:.6f} s")
print(f"\nOVERALL ACCURACY       : {100*sum(accs)/3:.2f}%")
print(f"OVERALL INFERENCE TIME : {sum(per_s)/3:.6f} seconds per sample")

RuleTaker     accuracy = 99.98%   time/sample = 0.000977 s
FOLIO         accuracy = 83.74%   time/sample = 0.047530 s
CounterBench  accuracy = 94.17%   time/sample = 0.000194 s

OVERALL ACCURACY       : 92.63%
OVERALL INFERENCE TIME : 0.016234 seconds per sample
